# 活产率分析
## 1. Fair-quality 胚胎好坏比较（活产率）
## 2. Logistic 回归分析（活产率）

In [65]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

print('✅ 库导入完成')

✅ 库导入完成


In [66]:
# 加载数据
df_viz = pd.read_csv('/Users/lzz/Documents/GitHub/IVF_AI/prediction/code/Embryo_FullStats_withQuality.csv')
df_patient_full = pd.read_csv('/Users/lzz/Downloads/df_fair_no_nan.csv')
df_pred = pd.read_csv('/Users/lzz/Downloads/embryo_results_2025-12-10.csv')
df_maindb = pd.read_csv('/Users/lzz/Documents/GitHub/IVF_AI/prediction/code/MainDB.csv')

# 用 visualization.ipynb 的胚胎集合做基准
key_cols = ['女病人ID', 'ARTID', '移植胚胎存活细胞数1', '移植数', '妊娠结局', '胎儿数', '囊胚等级', 'Quality']
for c in key_cols:
    df_viz[c] = df_viz[c].astype(str)
    df_patient_full[c] = df_patient_full[c].astype(str)

df_patient = df_patient_full.merge(df_viz[key_cols].drop_duplicates(), on=key_cols, how='inner')

print(f'✅ 数据加载完成')
print(f'df_viz: {df_viz.shape}, df_patient: {df_patient.shape}, df_pred: {df_pred.shape}')

✅ 数据加载完成
df_viz: (2746, 11), df_patient: (440, 15), df_pred: (1207, 4)


## 数据预处理：计算活产信息

In [67]:
# 查看数据基本信息
print('妊娠结局分布:')
print(df_patient['妊娠结局'].value_counts())

print('\n胎儿数分布:')
print(df_patient['胎儿数'].value_counts(dropna=False))

# 创建活产标记：胎儿数 > 0 表示活产
df_patient['胎儿数'] = pd.to_numeric(df_patient['胎儿数'], errors='coerce')
df_patient['活产'] = (df_patient['胎儿数'] > 0).astype(int)

print('\n活产分布:')
print(df_patient['活产'].value_counts())


妊娠结局分布:
妊娠结局
临床妊娠    230
未妊娠     150
生化妊娠     58
异位妊娠      2
Name: count, dtype: int64

胎儿数分布:
胎儿数
0    307
1    131
2      2
Name: count, dtype: int64

活产分布:
活产
0    307
1    133
Name: count, dtype: int64


## 分析1：Fair-quality 胚胎好坏比较（活产率）

In [68]:
# 合并预测数据和 MainDB（添加质量控制）
# Step 1: 提取 df_pred 的基础 ID
df_pred['base_id'] = df_pred['embryo_id'].str.rsplit('_WELL', n=1).str[0]

# Step 1.1: 构造与 MainDB 匹配的 RowName
# MainDB.RowName 形如: D2022.06.18_S00144_I4424_P_11
# df_pred.embryo_id 形如: D2022.06.18_S00144_I4424_P_WELL11
df_pred['RowName'] = df_pred['embryo_id'].str.replace('_WELL', '_', regex=False)

# Step 2: 合并预测结果
df_merged = df_patient.merge(df_pred, left_on='SlideID', right_on='base_id', how='left')

# Step 3: 合并 MainDB（用于质量控制）
df_merged = df_merged.merge(
    df_maindb[['RowName', 'ICM', 'TE']],
    on='RowName',
    how='left'
)

print(f'合并后数据 shape: {df_merged.shape}')
print(f'有预测结果的记录数: {df_merged["prediction"].notna().sum()}')
print(f'成功匹配 MainDB 的记录数: {df_merged["ICM"].notna().sum()}')


合并后数据 shape: (1421, 24)
有预测结果的记录数: 1361
成功匹配 MainDB 的记录数: 1361


In [69]:
# 质量控制筛选
# 从囊胚等级中提取 ICM 和 TE
import re

def extract_grade_components(grade):
    """从囊胚等级中提取 ICM 和 TE"""
    if pd.isna(grade) or not isinstance(grade, str):
        return pd.Series([None, None])
    
    # 匹配格式如 "3BB", "4AA" 等
    match = re.match(r'^\d([A-C])([A-C])', grade)
    if match:
        return pd.Series([match.group(1), match.group(2)])
    return pd.Series([None, None])

df_merged[['Grade_ICM', 'Grade_TE']] = df_merged['囊胚等级'].apply(extract_grade_components)

# 清理 MainDB 的 ICM/TE（转大写并去空格）
df_merged['ICM_clean'] = df_merged['ICM'].astype(str).str.upper().str.strip()
df_merged['TE_clean'] = df_merged['TE'].astype(str).str.upper().str.strip()

print(f'提取等级信息完成')


提取等级信息完成


In [70]:
# 应用质量控制筛选条件
# 筛选条件：
# 1. Fair-quality 胚胎（visualization 口径：3BB/4BB/5BB）
# 2. 有预测结果
# 3. 单胚胎移植（移植数=1）
# 4. ICM/TE 匹配（质量控制）

# 统一清理字段，避免类型/空格导致筛选失败
df_merged['Quality_clean'] = df_merged['Quality'].astype(str).str.strip()
df_merged['移植数_num'] = pd.to_numeric(df_merged['移植数'], errors='coerce')

df_fair = df_merged[
    (df_merged['Quality_clean'] == 'Fair-quality') &
    (df_merged['prediction'].notna()) &
    (df_merged['移植数_num'] == 1) &
    (df_merged['Grade_ICM'] == df_merged['ICM_clean']) &
    (df_merged['Grade_TE'] == df_merged['TE_clean'])
].copy()

print(f'✅ 质量控制筛选完成')
print(f'筛选后的 Fair-quality 单胚胎移植记录数: {len(df_fair)}')
print(f'\nGood/Poor 预测分布:')
print(df_fair['prediction'].value_counts())


✅ 质量控制筛选完成
筛选后的 Fair-quality 单胚胎移植记录数: 547

Good/Poor 预测分布:
prediction
Good    293
Poor    254
Name: count, dtype: int64


In [71]:
# 计算 Good vs Poor 的活产率
live_birth_rate = (
    df_fair
    .groupby('prediction')
    .agg({
        '活产': ['sum', 'count']
    })
    .reset_index()
)

live_birth_rate.columns = ['prediction', '活产数', '总数']
live_birth_rate['活产率(%)'] = live_birth_rate['活产数'] / live_birth_rate['总数'] * 100

print('===== Fair-quality 胚胎 Good vs Poor 活产率（移植数=1）=====')
print(live_birth_rate)
print(f'\n活产率差异: {live_birth_rate.iloc[0]["活产率(%)"] - live_birth_rate.iloc[1]["活产率(%)"]:.2f}%')

===== Fair-quality 胚胎 Good vs Poor 活产率（移植数=1）=====
  prediction  活产数   总数     活产率(%)
0       Good  109  293  37.201365
1       Poor   79  254  31.102362

活产率差异: 6.10%


In [72]:
# 统计检验：卡方检验
contingency_table = pd.crosstab(df_fair['prediction'], df_fair['活产'])
print('列联表:')
print(contingency_table)

chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print(f'\n卡方检验结果:')
print(f'Chi2 = {chi2:.4f}')
print(f'P-value = {p_value:.4f}')
print(f'显著性: {"显著" if p_value < 0.05 else "不显著"} (α=0.05)')

列联表:
活产            0    1
prediction          
Good        184  109
Poor        175   79

卡方检验结果:
Chi2 = 1.9814
P-value = 0.1592
显著性: 不显著 (α=0.05)


In [ ]:
# 可视化：Fair-quality 胚胎 Good vs Poor 活产率对比
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['#4CAF50', '#F44336']  # Green for Good, Red for Poor
bars = ax.bar(live_birth_rate['prediction'], live_birth_rate['活产率(%)'], 
              color=colors, edgecolor='white', width=0.6)

# 添加数值标签
for i, row in live_birth_rate.iterrows():
    ax.text(i, row['活产率(%)'] + 1, 
            f"{row['活产率(%)']:.1f}%\n(n={int(row['总数'])})", 
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('活产率 (%)', fontsize=12)
ax.set_xlabel('预测类别', fontsize=12)
ax.set_title('Fair-quality 胚胎：Good vs Poor 活产率对比\n(单胚胎移植)', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_ylim(0, max(live_birth_rate['活产率(%)']) * 1.2)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('fair_quality_live_birth_rate_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()

print('✅ 图表已保存为 fair_quality_live_birth_rate_comparison.pdf')

## 分析2：Logistic 回归分析（活产率）

In [ ]:
# Logistic回归（活产结局），控制Age后计算各指标OR值和95%CI
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm

FIG_DIR = "/Users/lzz/Documents/GitHub/IVF_AI/Figures"

# 1) 使用 visualization 生成的全量表（含 Quality）
if "df_viz" not in globals():
    df_viz = pd.read_csv('/Users/lzz/Documents/GitHub/IVF_AI/prediction/code/Embryo_FullStats_withQuality.csv')

df_filtered = df_viz.copy()

# 2) 仅保留 Good/Poor（与 visualization logistic 一致）
df_filtered = df_filtered[df_filtered["Quality"].isin(["Good-quality", "Poor-quality"])].copy()

# 3) 合并病人级 PC1 指标
pc1_path = "/Users/lzz/Documents/GitHub/IVF_AI/prediction/code/STORK_lzz_torch/outputs/pc1_cycle_vgg16/pc1_cycle_level.csv"
df_pc1 = pd.read_csv(pc1_path)
df_pc1["patient_id"] = pd.to_numeric(df_pc1["patient_id"], errors="coerce")

df_filtered["女病人ID"] = pd.to_numeric(df_filtered["女病人ID"], errors="coerce")
merged_q = df_filtered.merge(df_pc1, left_on="女病人ID", right_on="patient_id", how="inner")

# 4) 变量准备
merged_q["Age"] = pd.to_numeric(merged_q["年龄"], errors="coerce")

# 活产结局（由胎儿数生成）
merged_q["胎儿数"] = pd.to_numeric(merged_q["胎儿数"], errors="coerce")
merged_q["LiveBirth"] = (merged_q["胎儿数"] > 0).astype(int)

# Quality 二分类（Good=1, Poor=0）
merged_q["Quality"] = merged_q["Quality"].map({"Good-quality": 1, "Poor-quality": 0})

metrics = ["pc1_p90", "pc1_max", "pc1_mean", "pc1_std", "pc1_median", "n_embryos"]

# 变量名映射（用于显示）
metric_names = {
    "pc1_mean": "QCPmean",
    "pc1_p90": "QCP90",
    "pc1_median": "QCPmedian",
    "pc1_max": "QCPmax",
    "pc1_std": "QCPsd",
    "n_embryos": "Nembryos"
}

# 标准化连续变量
def zscore(s):
    return (s - s.mean()) / s.std(ddof=0)

# ========== 1) 特征OR图（控制Age后，带95%CI） ==========
full_base = merged_q[["LiveBirth", "Age", "Quality"] + metrics].dropna().copy()
if full_base.empty or full_base["LiveBirth"].nunique() < 2:
    print("Logistic回归训练失败：样本为空或活产单一类别")
else:
    full_base["Age"] = zscore(full_base["Age"])
    for m in metrics:
        full_base[m] = zscore(full_base[m])

    y_full = full_base["LiveBirth"].astype(int)

    print(f'用于Logistic回归分析的样本数: {len(full_base)}')
    print(f'活产数: {y_full.sum()}, 未活产数: {(1-y_full).sum()}')

    # 对每个PC指标，分别训练 Age + 该指标 的模型，并获取95%CI
    or_results = []
    
    print('\n计算各指标的OR值和95%CI（控制Age）:')
    print('='*60)
    
    for metric in metrics:
        # 使用statsmodels获取置信区间
        X = full_base[["Age", metric]]
        X_with_const = sm.add_constant(X)
        
        logit_model = sm.Logit(y_full, X_with_const)
        result = logit_model.fit(disp=0)
        
        # 获取该指标的系数、OR值和95%CI
        coef = result.params[metric]
        or_val = np.exp(coef)
        ci_lower = np.exp(result.conf_int().loc[metric, 0])
        ci_upper = np.exp(result.conf_int().loc[metric, 1])
        
        or_results.append({
            'metric': metric,
            'display_name': metric_names[metric],
            'or': or_val,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper
        })
        
        print(f'{metric_names[metric]:12s}: OR={or_val:.3f}, 95%CI=[{ci_lower:.3f}, {ci_upper:.3f}]')
    
    # 转换为DataFrame并排序
    or_df = pd.DataFrame(or_results)
    or_df = or_df.sort_values('or').reset_index(drop=True)
    
    # 绘制森林图（Forest plot）
    fig, ax = plt.subplots(figsize=(8, 5))
    
    y_pos = np.arange(len(or_df))
    
    # 绘制点和误差线
    for i, row in or_df.iterrows():
        ax.plot([row['ci_lower'], row['ci_upper']], [i, i], 
                color='#5B9AA0', linewidth=2, solid_capstyle='round')
        ax.plot(row['or'], i, 'o', color='#5B9AA0', markersize=8)
    
    # 设置y轴
    ax.set_yticks(y_pos)
    ax.set_yticklabels(or_df['display_name'].values, fontsize=12)
    
    # 设置x轴
    ax.set_xlabel('Odds Ratio', fontsize=12)
    
    # 添加中性线（OR=1）
    ax.axvline(x=1, color='gray', linestyle='--', linewidth=1.5)
    
    # 添加边框
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    ax.spines['left'].set_visible(True)
    ax.spines['bottom'].set_visible(True)
    
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/logistic_coefficients_live_birth.pdf", dpi=300, bbox_inches='tight')
    plt.close()
    print('\n✅ 特征OR图已保存（森林图，控制Age后的OR值和95%CI）')

# ========== 2) AUC 条形图（使用5折交叉验证） ==========
base = merged_q[["LiveBirth", "Age", "Quality"] + metrics].dropna().copy()
if base.empty or base["LiveBirth"].nunique() < 2:
    print("AUC 计算失败：样本为空或活产单一类别")
else:
    base["Age"] = zscore(base["Age"])
    for m in metrics:
        base[m] = zscore(base[m])

    y = base["LiveBirth"].astype(int)

    print(f'\n开始计算交叉验证AUC（5折交叉验证，Logistic回归）...')
    
    def auc_for(cols, model_name=""):
        """使用5折交叉验证计算AUC"""
        X = base[cols]
        clf = LogisticRegression(
            penalty='l2',
            C=1.0,
            max_iter=1000,
            random_state=42
        )
        pred = cross_val_predict(clf, X, y, cv=5, method='predict_proba')[:, 1]
        auc = roc_auc_score(y, pred)
        if model_name:
            print(f'  {model_name}: AUC = {auc:.4f}')
        return auc

    auc_age = auc_for(["Age"], "Age")
    auc_q = auc_for(["Age", "Quality"], "Age+Quality")

    rows = [
        {"Model": "Age", "AUC": auc_age},
        {"Model": "Age+Quality", "AUC": auc_q},
    ]

    rows.append({
        "Model": "Age+pc1_p90+Quality",
        "AUC": auc_for(["Age", "pc1_p90", "Quality"], "Age+pc1_p90+Quality")
    })

    for m in metrics:
        auc_pc = auc_for(["Age", m], f"Age+{m}")
        rows.append({"Model": f"Age+{m}", "AUC": auc_pc})

    auc_df = pd.DataFrame(rows).sort_values("AUC", ascending=False)

    print('\n===== 交叉验证AUC结果（5折CV，Logistic回归）=====')
    print(auc_df.to_string(index=False))

    plt.figure(figsize=(7, 4))
    plt.bar(auc_df["Model"], auc_df["AUC"], color='#A0C4FF')
    plt.ylim(0.5, 0.75)
    plt.ylabel('AUC (5-Fold CV)')
    plt.title('AUC Comparison (Live Birth, Logistic Regression, 5-Fold CV)')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/auc_comparison_live_birth_logistic.pdf")
    plt.close()
    print('\n✅ AUC比较图已保存')


In [74]:
# 已合并到上一单元：Logistic 回归（活产结局）


===== Logistic 回归结果（活产率）=====
Prediction (Good vs Poor) OR: 1.3933
Age OR: 0.6104
BMI OR: 0.8605

Model AUC: 0.6394
